# Build Groups C & D

In [8]:
import pandas as pd
import os

# --- Paths ---
elpac_dir = "../1095 Data/ELPAC/"

# --- Load Initial ELPAC files ---
initial_files = {
    "Initial 2022": "InitialElpac_22.csv",
    "Initial 2023": "InitialElpac_23.csv",
    "Initial 2024": "InitialElpac_24.csv",
}

initial_dfs = {}
for label, filename in initial_files.items():
    df = pd.read_csv(os.path.join(elpac_dir, filename))
    df = df.apply(lambda col: col.str.strip() if col.dtype == "object" else col)
    df["TestStartDate"] = pd.to_datetime(df["TestStartDate"], errors="coerce")
    df["OverallScaleScore"] = pd.to_numeric(df["OverallScaleScore"], errors="coerce")
    initial_dfs[label] = df
    print(f"{label}: {len(df):,} rows")

# --- Load Summative ELPAC files ---
summative_files = {
    1: "SummativeElpac_22.csv",
    2: "SummativeElpac_23.csv",
    3: "SummativeElpac_24.csv",
}

summative_dfs = {}
for year_num, filename in summative_files.items():
    df = pd.read_csv(os.path.join(elpac_dir, filename))
    df = df.apply(lambda col: col.str.strip() if col.dtype == "object" else col)
    df["OverallScaleScore"] = pd.to_numeric(df["OverallScaleScore"], errors="coerce")
    df["TestStartDate"] = pd.to_datetime(df["TestStartDate"], errors="coerce")
    summative_dfs[year_num] = df
    print(f"Summative {year_num}: {len(df):,} rows")

# --- Build initial record per student: earliest date and OverallPL ---
initial_records = {}  # SSID -> {"date": ..., "pl": ...}
for label in ["Initial 2022", "Initial 2023", "Initial 2024"]:
    df = initial_dfs[label][["SSID", "TestStartDate", "OverallPL"]].dropna()
    for _, row in df.iterrows():
        ssid = row["SSID"]
        if ssid not in initial_records or row["TestStartDate"] < initial_records[ssid]["date"]:
            initial_records[ssid] = {
                "date": row["TestStartDate"],
                "pl":   row["OverallPL"]
            }

print(f"\nStudents with initial records: {len(initial_records):,}")

# --- Build summative records per student: list of (date, score) tuples ---
summative_records = {}  # SSID -> list of (date, score)
for year_num, df in summative_dfs.items():
    valid = df[["SSID", "TestStartDate", "OverallScaleScore"]].dropna()
    for _, row in valid.iterrows():
        ssid = row["SSID"]
        if ssid not in summative_records:
            summative_records[ssid] = []
        summative_records[ssid].append((row["TestStartDate"], row["OverallScaleScore"]))

# --- Filter function: apply Group C/D criteria ---
def build_group(target_pl, score_threshold=1150):
    group_ssids = []
    for ssid, init in initial_records.items():
        # Must match target PL
        if init["pl"] != target_pl:
            continue
        # Must have summative records
        if ssid not in summative_records:
            continue
        # Keep only summative records after initial date and above score threshold
        valid_summatives = [
            (date, score) for date, score in summative_records[ssid]
            if date > init["date"] and score > score_threshold
        ]
        # Must have exactly 3 qualifying summative records
        if len(valid_summatives) == 3:
            group_ssids.append(ssid)
    return group_ssids

# --- Build Groups C and D ---
group_c_ssids = build_group("Novice English Learner")
group_d_ssids = build_group("Intermediate English Learner")

# print(f"\nGroup C (Novice, 3 summatives after initial date, all scores > 1150): {len(group_c_ssids):,} students")
# print(f"Group D (Intermediate, 3 summatives after initial date, all scores > 1150): {len(group_d_ssids):,} students")

# # --- Export to CSV ---
# pd.DataFrame({"SSID": group_c_ssids}).to_csv("group_c_ssids.csv", index=False)
# pd.DataFrame({"SSID": group_d_ssids}).to_csv("group_d_ssids.csv", index=False)

# print("\nSaved: group_c_ssids.csv")
# print("Saved: group_d_ssids.csv")

Initial 2022: 71,083 rows
Initial 2023: 71,637 rows
Initial 2024: 76,539 rows
Summative 1: 332,779 rows
Summative 2: 319,636 rows
Summative 3: 313,603 rows

Students with initial records: 196,144


In [9]:
# --- Step 1: Check OverallPL values (whitespace issues?) ---
print("=== Step 1: Unique OverallPL values in initial_records ===")
pl_counts = {}
for ssid, rec in initial_records.items():
    pl = rec["pl"]
    pl_counts[pl] = pl_counts.get(pl, 0) + 1
for pl, count in sorted(pl_counts.items(), key=lambda x: -x[1]):
    print(f"  '{pl}': {count:,}")

# --- Step 2: Check summative score distribution ---
print("\n=== Step 2: Summative OverallScaleScore range ===")
for year_num, df in summative_dfs.items():
    scores = df["OverallScaleScore"].dropna()
    print(f"  Summative {year_num}: min={scores.min():.0f}, max={scores.max():.0f}, "
          f"mean={scores.mean():.0f}, >1150: {(scores > 1150).sum():,}")

# --- Step 3: Check how many students have ANY summative after initial date ---
print("\n=== Step 3: Summative date vs initial date ===")
after_count = 0
no_initial_date = 0
for ssid, summ_list in summative_records.items():
    if ssid not in initial_records:
        continue
    init_date = initial_records[ssid]["date"]
    if pd.isna(init_date):
        no_initial_date += 1
        continue
    if any(date > init_date for date, score in summ_list):
        after_count += 1
print(f"  Students with at least 1 summative after initial date: {after_count:,}")
print(f"  Students with missing initial date: {no_initial_date:,}")

# --- Step 4: Check how many qualifying summatives each student has ---
print("\n=== Step 4: Count of valid summatives (after initial date AND score > 1150) ===")
valid_counts = {}
for ssid, init in initial_records.items():
    if init["pl"].strip() != "Novice English Learner":
        continue
    if ssid not in summative_records:
        continue
    init_date = init["date"]
    if pd.isna(init_date):
        continue
    valid = [
        (date, score) for date, score in summative_records[ssid]
        if date > init_date and score > 1150
    ]
    n = len(valid)
    valid_counts[n] = valid_counts.get(n, 0) + 1

print("  Novice students by number of qualifying summatives:")
for n, count in sorted(valid_counts.items()):
    print(f"    {n} qualifying summatives: {count:,} students")

# --- Step 5: Check for duplicate summative records per student per year ---
print("\n=== Step 5: Summative records per student (are there duplicates?) ===")
for year_num, df in summative_dfs.items():
    dupes = df.groupby("SSID").size()
    print(f"  Summative {year_num}: max records per student={dupes.max()}, "
          f"students with >1 record: {(dupes > 1).sum():,}")

=== Step 1: Unique OverallPL values in initial_records ===
  'Novice English Learner                  ': 149,342
  'Intermediate English Learner            ': 24,809
  'Initial Fluent English Proficient (IFEP)': 21,364
  'NS                                      ': 629

=== Step 2: Summative OverallScaleScore range ===
  Summative 1: min=201, max=1950, mean=1474, >1150: 310,211
  Summative 2: min=201, max=1950, mean=1471, >1150: 303,478
  Summative 3: min=201, max=1868, mean=1463, >1150: 297,309

=== Step 3: Summative date vs initial date ===
  Students with at least 1 summative after initial date: 165,819
  Students with missing initial date: 0

=== Step 4: Count of valid summatives (after initial date AND score > 1150) ===
  Novice students by number of qualifying summatives:
    0 qualifying summatives: 3,833 students
    1 qualifying summatives: 58,276 students
    2 qualifying summatives: 43,514 students
    3 qualifying summatives: 28,953 students
    4 qualifying summatives: 5,65

In [6]:
# --- Rebuild summative records: one record per student per year (highest score) ---
summative_records = {}  # SSID -> {year_num: (date, score)}
for year_num, df in summative_dfs.items():
    valid = df[["SSID", "TestStartDate", "OverallScaleScore"]].dropna()
    # Keep highest score per student for this year
    best = valid.loc[valid.groupby("SSID")["OverallScaleScore"].idxmax()]
    for _, row in best.iterrows():
        ssid = row["SSID"]
        if ssid not in summative_records:
            summative_records[ssid] = {}
        summative_records[ssid][year_num] = (row["TestStartDate"], row["OverallScaleScore"])

print(f"Students with at least 1 summative record: {len(summative_records):,}")

# --- Fixed build_group: strip whitespace, require one qualifying record per year ---
def build_group(target_pl, score_threshold=1150):
    group_ssids = []
    for ssid, init in initial_records.items():
        # Strip whitespace when comparing
        if init["pl"].strip() != target_pl:
            continue
        if ssid not in summative_records:
            continue
        init_date = init["date"]
        # Check all 3 years have a qualifying record
        qualifying_years = 0
        for year_num in [1, 2, 3]:
            if year_num not in summative_records[ssid]:
                continue
            date, score = summative_records[ssid][year_num]
            if date > init_date and score > score_threshold:
                qualifying_years += 1
        if qualifying_years == 3:
            group_ssids.append(ssid)
    return group_ssids

# --- Build Groups C and D ---
group_c_ssids = build_group("Novice English Learner")
group_d_ssids = build_group("Intermediate English Learner")

print(f"\nGroup C (Novice, 3 summatives after initial date, all scores > 1150):        {len(group_c_ssids):,} students")
print(f"Group D (Intermediate, 3 summatives after initial date, all scores > 1150):  {len(group_d_ssids):,} students")

# --- Export to CSV ---
pd.DataFrame({"SSID": group_c_ssids}).to_csv("group_c_ssids.csv", index=False)
pd.DataFrame({"SSID": group_d_ssids}).to_csv("group_d_ssids.csv", index=False)

print("\nSaved: group_c_ssids.csv")
print("Saved: group_d_ssids.csv")

Students with at least 1 summative record: 426,676

Group C (Novice, 3 summatives after initial date, all scores > 1150):        28,055 students
Group D (Intermediate, 3 summatives after initial date, all scores > 1150):  4,194 students

Saved: group_c_ssids.csv
Saved: group_d_ssids.csv
